# Lab 5 - More Exploratory Data Analytics

In [1]:
import sys
!"{sys.executable}" -m pip install pandas
!"{sys.executable}" -m pip install numpy
!"{sys.executable}" -m pip install matplotlib
!"{sys.executable}" -m pip install seaborn
!"{sys.executable}" -m pip install scipy
!"{sys.executable}" -m pip install scikit-learn
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import scipy

For this lab, we are going to use the Palmer Penguins dataset. The dataset contains characteristics of 344 penguins, including their species, flipper length, and body mass. This dataset can be found on the Moodle, and can be imported as follows:

In [2]:
penguins = pd.read_csv("penguins.csv")

penguins.head()

,species,island,culmen_length_mm,culmen_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,MALE
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,FEMALE
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,FEMALE
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,FEMALE


Let's explore this data. First, have a look through the data in your favourite CSV viewer (e.g. Microsoft Excel). Do you notice anything about the data? Let's now programatically explore the data. We can look for missing data by using the isnull function.

In [3]:
penguins.isnull()

,species,island,culmen_length_mm,culmen_depth_mm,flipper_length_mm,body_mass_g,sex
0,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False
3,False,False,True,True,True,True,True
4,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...
339,False,False,True,True,True,True,True
340,False,False,False,False,False,False,False
341,False,False,False,False,False,False,False
342,False,False,False,False,False,False,False


This may not be especially useful by itself, but if we take a quick look, we can see a small number of "True" values, indicating missing data. We can take the sum of this data matrix to determine the total number of missing values.

In [4]:
print("Missing values by attribute:\n", penguins.isnull().sum())
print("Total number of missing values: ", penguins.isnull().sum().sum())

Missing values by attribute:
 species               0
island                0
culmen_length_mm      2
culmen_depth_mm       2
flipper_length_mm     2
body_mass_g           2
sex                  10
dtype: int64
Total number of missing values:  18


How shall we handle these missing values? As discussed in lectures, there are many ways for us to do this. The first would be to simply delete all objects which contain missing data. This can be achieved as follows:

In [3]:
# 删除所有包含缺失值的行，并将清洗后的DataFrame保存为penguins_clean
penguins_clean = penguins.dropna()

# 查看原始DataFrame的结构、数据类型及非缺失值数量
penguins.info()

# 输出分隔线，方便区分清洗前后的信息
print("============")

# 查看清洗后的DataFrame信息
penguins_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 344 entries, 0 to 343
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   species            344 non-null    str    
 1   island             344 non-null    str    
 2   culmen_length_mm   342 non-null    float64
 3   culmen_depth_mm    342 non-null    float64
 4   flipper_length_mm  342 non-null    float64
 5   body_mass_g        342 non-null    float64
 6   sex                334 non-null    str    
dtypes: float64(4), str(3)
memory usage: 18.9 KB
<class 'pandas.DataFrame'>
Index: 334 entries, 0 to 343
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   species            334 non-null    str    
 1   island             334 non-null    str    
 2   culmen_length_mm   334 non-null    float64
 3   culmen_depth_mm    334 non-null    float64
 4   flipper_length_mm  334 non-null    float64
 5

penguins_clean now contains 334 objects with no missing data. What if we didn't want to simply delete these objects though? Could we perhaps infer the missing values?

Let's focus just on the _body_mass_g_ attribute. It is the body mass of the penguin, in grams. Let's try a few methods to fill in the missing values. Firstly, we can simply fill in the missing values with the mean of the attribute. There is a very useful method known as fillna() which allows us to fill in missing values.

In [4]:
# 复制原始penguins数据集，避免修改原始数据
penguins_copy = penguins.copy()

# 计算并打印body_mass_g列的平均值，默认忽略NaN
print("The mean value of the \"body_mass_g\" attribute is", penguins_copy["body_mass_g"].mean())

# 统计填补前body_mass_g列的缺失值数量
print("Total number of missing values in the \"body_mass_g\" attribute before imputing: ", penguins_copy["body_mass_g"].isnull().sum())

# .fillna(value)：
# 将Series中的缺失值替换为指定的value
# 使用body_mass_g列的平均值填补该列中的所有缺失值
penguins_copy["body_mass_g"] = penguins_copy["body_mass_g"].fillna(penguins_copy["body_mass_g"].mean())

# 统计填补后body_mass_g列剩余的缺失值数量
print("Total number of missing values in the \"body_mass_g\" attribute after imputing: ", penguins_copy["body_mass_g"].isnull().sum())

The mean value of the "body_mass_g" attribute is 4201.754385964912
Total number of missing values in the "body_mass_g" attribute before imputing:  2
Total number of missing values in the "body_mass_g" attribute after imputing:  0


We can put anything in fillna, even zero (although we'd need a good reason to do this...):

In [5]:
penguins_copy = penguins.copy()

penguins_copy["body_mass_g"] = penguins_copy["body_mass_g"].fillna(0)

As mentioned in the lectures, we _could_ use a technique such as regression to fill in the missing values.

Regression works by utilising the values of other attributes to estimate the value of a specific target attribute. We could consider using a regression model to predict the value of body_mass_g using culmen_length_mm, culmen_depth_mm, and flipper_length_mm. However, in our original penguins CSV, the two objects with missing body_mass_g values also have missing values for culmen_length_mm, culmen_depth_mm, and flipper_length_mm. Hence, we would not be able to predict the value of body_mass_g because the predictor attributes are missing. *Solely for the purposes of demosntrating how a regression model（回归模型） can be used in Python*, we will introduce penguins_2, a slightly modified version of penguins which provides some values of culmen_length_mm, culmen_depth_mm, and flipper_length_mm to the objects with missing body_mass_g, to allow for a regression model to be used.

We can use a regression model as follows:

In [ ]:
# 导入Scikit-learn的线性回归模型
from sklearn import linear_model

# 导入训练集与测试集划分函数（本段代码暂未使用）
from sklearn.model_selection import train_test_split

# 读取包含缺失值的企鹅数据集
penguins_copy = pd.read_csv("penguins_2.csv")

# 删除包含缺失值的行，作为模型训练数据
# 注意：这里使用的是penguins，而不是刚读取的penguins_copy
penguins_for_regression = penguins.dropna()

# 选择3个身体特征作为自变量X
X_train = penguins_for_regression[["culmen_length_mm", "culmen_depth_mm", "flipper_length_mm"]]

# 选择企鹅体重作为因变量y
y_train = penguins_for_regression["body_mass_g"]

# 创建线性回归模型
lin_regr = linear_model.LinearRegression()

# 利用已知身体特征和体重训练模型
lin_regr.fit(X_train, y_train)

# 筛选body_mass_g缺失的企鹅记录
# penguins_copy["body_mass_g"].isna()：
# 判断每一行的体重是否缺失
# 缺失返回True，非缺失返回False
missing_penguins = penguins_copy[penguins_copy["body_mass_g"].isna()]

# 提取需要预测体重的企鹅身体特征
X_missing = missing_penguins[["culmen_length_mm", "culmen_depth_mm", "flipper_length_mm"]]

# 使用训练好的模型预测缺失体重
predicted_body_mass_g = lin_regr.predict(X_missing)

# 将预测结果填回原来body_mass_g缺失的位置
penguins_copy.loc[penguins_copy["body_mass_g"].isna(), "body_mass_g"] = predicted_body_mass_g

Remember: If you are filling in a missing value using a technique such as regression, you are relying on other attributes to use to determine your missing value!

As regression would not be appropriate in this case, lets just fill in missing values with the mean of the attribute:

In [6]:
penguins = penguins.fillna(penguins.select_dtypes(include="number").mean())
penguins.head()

,species,island,culmen_length_mm,culmen_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.10000,18.70000,181.000000,3750.000000,MALE
1,Adelie,Torgersen,39.50000,17.40000,186.000000,3800.000000,FEMALE
2,Adelie,Torgersen,40.30000,18.00000,195.000000,3250.000000,FEMALE
3,Adelie,Torgersen,43.92193,17.15117,200.915205,4201.754386,NaN
4,Adelie,Torgersen,36.70000,19.30000,193.000000,3450.000000,FEMALE


You will see from the above example that the fourth row (id=3) has had its numeric attributes replaced with the mean of the attribute. This does not mean we have removed all missing data, however! We can see that sex is missing for the fourth object too. Obviously we cannot impute an attribute such as this with the mean. In this case, let's drop any remaining rows with missing data.

In [7]:
penguins = penguins.dropna()
penguins.head()

,species,island,culmen_length_mm,culmen_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,MALE
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,FEMALE
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,FEMALE
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,FEMALE
5,Adelie,Torgersen,39.3,20.6,190.0,3650.0,MALE


Let's have a look at the data types of each attribute:

In [8]:
penguins.dtypes

species                  str
island                   str
culmen_length_mm     float64
culmen_depth_mm      float64
flipper_length_mm    float64
body_mass_g          float64
sex                      str
dtype: object

The above look OK, but there is a datatype known as Category. This is similar to string, but is more efficient, and ensures our data is recognised as consisting of a series of categories.

In [ ]:
# 定义需要转换为类别型数据的3个列名
categorical_columns = ["species", "island", "sex"]

# 选择这3列，并将数据类型统一转换为category
penguins[categorical_columns] = penguins[categorical_columns].astype("category")

# 查看DataFrame中所有列的数据类型
penguins.dtypes

species              category
island               category
culmen_length_mm      float64
culmen_depth_mm       float64
flipper_length_mm     float64
body_mass_g           float64
sex                  category
dtype: object

Now our data is ready, let's explore!

Let's propose a few questions, and then try to answer them by exploring the data. Firstly, let's start with an easy one, which penguin species is the most common? We can answer that in code, but let's also generate a visualisation.

Value_counts gives us counts of unique values within our data.

In [ ]:
penguins["species"].value_counts()

At it's most basic level...

In [ ]:
sns.countplot(data=penguins, x="species")

Q1: Improve the above visualisation, by adding extra characteristics to the plot (e.g. title).

Q2: Modify the above count plot to include the proportions of each species in the data with each sex. You may need to change the type of plot.

Now, does body mass differ for each species? Well, we could use a boxplot here:

In [ ]:
sns.boxplot(data=penguins, x='species', y='body_mass_g')

The answer is clearly yes! The Gentoo species is much heavier, in most cases! Let's now see whether flipper length and body mass are related.

In [ ]:
sns.scatterplot(data=penguins, x='flipper_length_mm', y='body_mass_g')

Yes! And there is quite a strong positive correlation between the two! Let's check what the correlation actually is...

In [ ]:
correlation = penguins.select_dtypes(include="number").corr()
sns.heatmap(correlation, cmap="flare", annot=True)

The above confirms what we saw in our scatterplot, but to be 100% sure, let's create another scatterplot, this time with culmen_depth_mm and culmen_length_mm. We're expecting a weak negative correlation (e.g. a x increases, y tends to decrease, although the relationship is not strong).

In [ ]:
sns.scatterplot(data=penguins, x='culmen_depth_mm', y='culmen_length_mm')

Maybe this is showing a weak negative correlation, but it's not obvious. Let's try a regplot, which is simply a scatterplot with a regression line.

In [ ]:
sns.regplot(data=penguins, x='culmen_depth_mm', y='culmen_length_mm')

Now our regression line is added, our weak negative correlation is much clearer! Next, let's explore the skew of our data, specifically with body_mass_g. We can calculate this like so.

In [ ]:
penguins["body_mass_g"].skew()

So a moderate positive skew. This tells us that most of our data is on the lower end, but with a long tail extending to the right. Let's visualise our data and see if it agrees with what we've just discovered.

In [ ]:
sns.histplot(data=penguins, x="body_mass_g")

Yes, it does!

Now, let's ask one more question - Which pairs of measurements are most useful for distinguising between species?

In [ ]:
sns.pairplot(penguins, hue="species")

Here, we can reasonably conclude that the best pair of attributes for distinguishing between species is culmen_depth_mm and culmen_length_mm, or flipper_length_mm and culmen_length_mm.

Q3. Propose three questions or hypotheses using the Penguins dataset, and perform EDA to try to answer those questions.  
Q4. Visit Kaggle, or your favourite data repository. Find some data, and attempt to gather some insight about the data, by proposing questions and performing EDA. This is a good opportunity to start thinking about the dataset you'd like to use, and the area you'd like to work in, for your CS950 Coursework!

## Bibliography

1. Data published in: Gorman KB, Williams TD, Fraser WR (2014). Ecological sexual dimorphism and environmental variability within a community of Antarctic penguins (genus Pygoscelis). PLoS ONE 9(3):e90081. https://doi.org/10.1371/journal.pone.0090081
2. Dataset downloaded from: Kaggle, 2020. Palmer Archipelago (Antarctica) penguin data. Available at: https://www.kaggle.com/datasets/parulpandey/palmer-archipelago-antarctica-penguin-data/data (Accessed 07 November 2026).